# Maze Pathfinding with Transformer (Encoder-Decoder)

**COL 774 Assignment 4**

## 1. Setup Environment

In [ ]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Upload Data Files

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Please upload your CSV data files")
    uploaded = files.upload()

## 3. Install Dependencies

In [ ]:
# Install additional packages if needed
!pip install -q torch pandas matplotlib tqdm numpy

## 4. Configuration

In [ ]:
# Configuration settings for maze pathfinding model training

# Transformer Model hyperparameters (as specified in assignment)
TRANSFORMER_CONFIG = {
    'd_model': 128,              # D-MODEL
    'nhead': 8,                  # NHEAD  
    'num_layers': 6,             # NUM-LAYERS (both encoder and decoder)
    'dim_feedforward': 512,      # DIM-FEEDFORWARD
    'dropout': 0.1,              # DROPOUT
    'num_epochs': 20,
    'learning_rate': 1e-4,
    'max_seq_length': 1024
}

# Data paths
DATA_CONFIG = {
    'train_csv': 'train_6x6_mazes.csv',
    'test_csv': 'test_6x6_mazes.csv'
}

# Training configuration
TRAIN_CONFIG = {
    'batch_size': 32,
    'val_split': 0.1,
    'checkpoint_dir': 'checkpoints_transformer',
    'save_every': 5  # Save checkpoint every N epochs
}

import os
os.makedirs(TRAIN_CONFIG['checkpoint_dir'], exist_ok=True)

print("Configuration loaded ✓")
print(f"\nModel will train for {TRANSFORMER_CONFIG['num_epochs']} epochs")
print(f"Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"Learning rate: {TRANSFORMER_CONFIG['learning_rate']}")
print(f"\nTransformer architecture:")
print(f"  d_model: {TRANSFORMER_CONFIG['d_model']}")
print(f"  nhead: {TRANSFORMER_CONFIG['nhead']}")
print(f"  num_layers: {TRANSFORMER_CONFIG['num_layers']}")
print(f"  dim_feedforward: {TRANSFORMER_CONFIG['dim_feedforward']}")

## 5. Model Implementation

In [ ]:
import torch
import torch.nn as nn
import math


class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding from 'Attention Is All You Need' (Section 3.5)
    
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute division term: 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-math.log(10000.0) / d_model))
        
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Add batch dimension
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerPathfinder(nn.Module):
    """
    Transformer Encoder-Decoder for maze pathfinding
    
    Architecture:
    - Shared embedding layer (Section 3.4 of paper)
    - Positional encoding (Section 3.5)
    - Transformer encoder (multi-head self-attention)
    - Transformer decoder (masked self-attention + cross-attention)
    - Output projection
    """
    
    def __init__(self, vocab_size, d_model=128, nhead=8, 
                 num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=512, dropout=0.1, 
                 max_seq_length=1024, pad_idx=0):
        super(TransformerPathfinder, self).__init__()
        
        self.d_model = d_model
        self.pad_idx = pad_idx
        
        # Shared embedding layer (Section 3.4)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.embedding_scale = math.sqrt(d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_seq_length, dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers
        )
        
        # Output projection (shared weights with embedding)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.fc_out.weight = self.embedding.weight
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier initialization"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def generate_square_subsequent_mask(self, sz, device):
        """Generate causal mask for decoder (Section 3.2.3)"""
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def create_padding_mask(self, seq, pad_idx):
        """Create mask for padding tokens"""
        return (seq == pad_idx)
    
    def forward(self, src, tgt):
        """
        Forward pass
        
        Args:
            src: Source sequence (batch_size, src_len)
            tgt: Target sequence (batch_size, tgt_len)
        
        Returns:
            Output logits (batch_size, tgt_len, vocab_size)
        """
        device = src.device
        tgt_len = tgt.size(1)
        
        # Create masks
        tgt_mask = self.generate_square_subsequent_mask(tgt_len, device)
        src_padding_mask = self.create_padding_mask(src, self.pad_idx)
        tgt_padding_mask = self.create_padding_mask(tgt, self.pad_idx)
        
        # Encoder
        src_emb = self.embedding(src) * self.embedding_scale
        src_emb = self.pos_encoder(src_emb)
        memory = self.transformer_encoder(
            src_emb,
            src_key_padding_mask=src_padding_mask
        )
        
        # Decoder
        tgt_emb = self.embedding(tgt) * self.embedding_scale
        tgt_emb = self.pos_encoder(tgt_emb)
        decoder_output = self.transformer_decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        # Output projection
        output = self.fc_out(decoder_output)
        return output
    
    def generate(self, src, max_len=200, sos_idx=1, eos_idx=2):
        """Greedy decoding for inference"""
        self.eval()
        batch_size = src.size(0)
        device = src.device
        
        # Encode source
        src_padding_mask = self.create_padding_mask(src, self.pad_idx)
        src_emb = self.embedding(src) * self.embedding_scale
        src_emb = self.pos_encoder(src_emb)
        memory = self.transformer_encoder(src_emb, src_key_padding_mask=src_padding_mask)
        
        # Initialize with SOS
        tgt = torch.full((batch_size, 1), sos_idx, dtype=torch.long, device=device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        
        for _ in range(max_len - 1):
            tgt_len = tgt.size(1)
            tgt_mask = self.generate_square_subsequent_mask(tgt_len, device)
            
            tgt_emb = self.embedding(tgt) * self.embedding_scale
            tgt_emb = self.pos_encoder(tgt_emb)
            decoder_output = self.transformer_decoder(
                tgt_emb, memory, tgt_mask=tgt_mask,
                memory_key_padding_mask=src_padding_mask
            )
            
            output = self.fc_out(decoder_output[:, -1, :])
            next_token = output.argmax(dim=-1, keepdim=True)
            tgt = torch.cat([tgt, next_token], dim=1)
            
            finished = finished | (next_token.squeeze(-1) == eos_idx)
            if finished.all():
                break
        
        return tgt


print("Model classes defined ✓")

## 6. Data Loading

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import ast
from typing import List


class MazeTokenizer:
    """Tokenizer for maze token sequences"""
    
    def __init__(self):
        self.PAD_TOKEN = '<PAD>'
        self.SOS_TOKEN = '<SOS>'
        self.EOS_TOKEN = '<EOS>'
        self.UNK_TOKEN = '<UNK>'
        self.token2idx = {}
        self.idx2token = {}
        self.vocab_size = 0
    
    def build_vocab(self, sequences: List[List[str]]):
        special_tokens = [self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN, self.UNK_TOKEN]
        unique_tokens = set()
        for seq in sequences:
            for token in seq:
                if isinstance(token, str):
                    unique_tokens.add(token)
        
        vocab = special_tokens + sorted(list(unique_tokens - set(special_tokens)))
        self.token2idx = {token: idx for idx, token in enumerate(vocab)}
        self.idx2token = {idx: token for token, idx in self.token2idx.items()}
        self.vocab_size = len(vocab)
        
        print(f"Vocabulary built: {self.vocab_size} tokens")
    
    def encode(self, tokens: List[str]) -> List[int]:
        return [self.token2idx.get(token, self.token2idx[self.UNK_TOKEN]) for token in tokens]
    
    def decode(self, indices: List[int]) -> List[str]:
        return [self.idx2token[idx] for idx in indices if idx in self.idx2token]
    
    @property
    def pad_idx(self):
        return self.token2idx[self.PAD_TOKEN]
    
    @property
    def sos_idx(self):
        return self.token2idx[self.SOS_TOKEN]
    
    @property
    def eos_idx(self):
        return self.token2idx[self.EOS_TOKEN]


class MazeDataset(Dataset):
    """Dataset for maze pathfinding"""
    
    def __init__(self, csv_path: str, tokenizer: MazeTokenizer = None):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        
        # Parse string representations to lists
        self.df['input_sequence'] = self.df['input_sequence'].apply(ast.literal_eval)
        self.df['output_path'] = self.df['output_path'].apply(ast.literal_eval)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_tokens = row['input_sequence']
        output_tokens = row['output_path']
        
        # Encode tokens
        input_ids = self.tokenizer.encode(input_tokens)
        output_ids = self.tokenizer.encode(output_tokens)
        
        return {
            'input': torch.tensor(input_ids, dtype=torch.long),
            'output': torch.tensor(output_ids, dtype=torch.long),
            'maze_type': row['maze_type']
        }


def collate_fn(batch, tokenizer):
    """Collate function with sequence shifting for transformer"""
    inputs = [item['input'] for item in batch]
    outputs = [item['output'] for item in batch]
    
    # Pad input sequences
    src = nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=tokenizer.pad_idx)
    
    # Decoder input: <SOS> + output
    tgt_input = []
    for out in outputs:
        tgt_input.append(torch.cat([torch.tensor([tokenizer.sos_idx]), out]))
    tgt_input = nn.utils.rnn.pad_sequence(tgt_input, batch_first=True, padding_value=tokenizer.pad_idx)
    
    # Target: output + <EOS>
    tgt_output = []
    for out in outputs:
        tgt_output.append(torch.cat([out, torch.tensor([tokenizer.eos_idx])]))
    tgt_output = nn.utils.rnn.pad_sequence(tgt_output, batch_first=True, padding_value=tokenizer.pad_idx)
    
    return {
        'src': src,
        'tgt': tgt_input,
        'target': tgt_output
    }


print("Data loading utilities defined ✓")

## 7. Evaluation Metrics

In [ ]:
def token_accuracy(predictions, targets, pad_idx):
    """
    Calculate token-level accuracy (ignoring padding)
    """
    mask = (targets != pad_idx)
    correct = (predictions == targets) & mask
    if mask.sum() == 0:
        return 0.0
    return (correct.sum().float() / mask.sum().float()).item()


def sequence_accuracy(predictions, targets, pad_idx):
    """
    Calculate sequence-level accuracy (exact match)
    """
    mask = (targets != pad_idx)
    correct = (predictions == targets) & mask
    correct_sequences = (correct.sum(dim=1) == mask.sum(dim=1)).float()
    return correct_sequences.mean().item()


def compute_f1_score(predictions, targets, pad_idx):
    """
    Calculate F1-Score over predicted tokens
    """
    mask = (targets != pad_idx)
    pred_flat = predictions[mask]
    target_flat = targets[mask]
    
    if len(target_flat) == 0:
        return 0.0
    
    true_positives = (pred_flat == target_flat).sum().item()
    false_positives = len(pred_flat) - true_positives
    false_negatives = len(target_flat) - true_positives
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    
    if precision + recall == 0:
        return 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


print("Evaluation metrics defined ✓")

## 8. Load Data

In [ ]:
# Load training data
print("Loading training data...")
train_df = pd.read_csv(DATA_CONFIG['train_csv'])
train_df['input_sequence'] = train_df['input_sequence'].apply(ast.literal_eval)
train_df['output_path'] = train_df['output_path'].apply(ast.literal_eval)

# Build vocabulary
tokenizer = MazeTokenizer()
all_sequences = (
    train_df['input_sequence'].tolist() + 
    train_df['output_path'].tolist()
)
tokenizer.build_vocab(all_sequences)

print(f"Training samples: {len(train_df)}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"PAD idx: {tokenizer.pad_idx}")
print(f"SOS idx: {tokenizer.sos_idx}")
print(f"EOS idx: {tokenizer.eos_idx}")

# Create datasets
train_dataset = MazeDataset(DATA_CONFIG['train_csv'], tokenizer)
test_dataset = MazeDataset(DATA_CONFIG['test_csv'], tokenizer)

# Split train into train/val
from torch.utils.data import random_split
train_size = int((1 - TRAIN_CONFIG['val_split']) * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

print(f"\nData split:")
print(f"  Train: {len(train_subset)} samples")
print(f"  Val: {len(val_subset)} samples")
print(f"  Test: {len(test_dataset)} samples")

# Create data loaders
from functools import partial

train_loader = DataLoader(
    train_subset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    collate_fn=partial(collate_fn, tokenizer=tokenizer),
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_subset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
    collate_fn=partial(collate_fn, tokenizer=tokenizer),
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
    collate_fn=partial(collate_fn, tokenizer=tokenizer),
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print("\nData loaders created ✓")

## 9. Create Model

In [ ]:
# Create model
model = TransformerPathfinder(
    vocab_size=tokenizer.vocab_size,
    d_model=TRANSFORMER_CONFIG['d_model'],
    nhead=TRANSFORMER_CONFIG['nhead'],
    num_encoder_layers=TRANSFORMER_CONFIG['num_layers'],
    num_decoder_layers=TRANSFORMER_CONFIG['num_layers'],
    dim_feedforward=TRANSFORMER_CONFIG['dim_feedforward'],
    dropout=TRANSFORMER_CONFIG['dropout'],
    max_seq_length=TRANSFORMER_CONFIG['max_seq_length'],
    pad_idx=tokenizer.pad_idx
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*70)
print("MODEL SUMMARY")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nArchitecture:")
print(f"  Vocabulary size: {tokenizer.vocab_size}")
print(f"  d_model: {TRANSFORMER_CONFIG['d_model']}")
print(f"  nhead: {TRANSFORMER_CONFIG['nhead']}")
print(f"  num_layers: {TRANSFORMER_CONFIG['num_layers']}")
print(f"  dim_feedforward: {TRANSFORMER_CONFIG['dim_feedforward']}")
print("="*70)

# Setup optimizer and loss
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=TRANSFORMER_CONFIG['learning_rate']
)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)

print("\nOptimizer: Adam")
print(f"Learning rate: {TRANSFORMER_CONFIG['learning_rate']}")
print("Loss: CrossEntropyLoss (ignoring padding)")

## 10. Training Loop

In [ ]:
from tqdm.notebook import tqdm
import time


def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch using teacher forcing"""
    model.train()
    total_loss = 0
    total_token_acc = 0
    total_seq_acc = 0
    total_f1 = 0
    num_batches = 0
    
    progress_bar = tqdm(train_loader, desc="Training", leave=False)
    
    for batch in progress_bar:
        src = batch['src'].to(device)
        tgt = batch['tgt'].to(device)
        target = batch['target'].to(device)
        
        optimizer.zero_grad()
        output = model(src, tgt)
        
        # Compute loss
        output_flat = output.reshape(-1, output.size(-1))
        target_flat = target.reshape(-1)
        loss = criterion(output_flat, target_flat)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # Metrics
        predictions = output.argmax(dim=-1)
        token_acc = token_accuracy(predictions, target, tokenizer.pad_idx)
        seq_acc = sequence_accuracy(predictions, target, tokenizer.pad_idx)
        f1 = compute_f1_score(predictions, target, tokenizer.pad_idx)
        
        total_loss += loss.item()
        total_token_acc += token_acc
        total_seq_acc += seq_acc
        total_f1 += f1
        num_batches += 1
        
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'tok_acc': f'{token_acc:.4f}',
            'seq_acc': f'{seq_acc:.4f}'
        })
    
    return {
        'loss': total_loss / num_batches,
        'token_accuracy': total_token_acc / num_batches,
        'seq_accuracy': total_seq_acc / num_batches,
        'f1_score': total_f1 / num_batches
    }


def evaluate(model, val_loader, criterion, device, tokenizer):
    """Evaluate model without teacher forcing (but compute loss like RNN)"""
    model.eval()
    total_loss = 0
    total_token_acc = 0
    total_seq_acc = 0
    total_f1 = 0
    num_batches = 0
    
    progress_bar = tqdm(val_loader, desc="Evaluating", leave=False)
    
    with torch.no_grad():
        for batch in progress_bar:
            src = batch['src'].to(device)
            tgt = batch['tgt'].to(device)
            target = batch['target'].to(device)
            
            # Forward pass WITHOUT teacher forcing (model uses its own predictions)
            # But we still pass tgt for computing loss against ground truth
            output = model(src, tgt)
            
            # Compute loss
            output_flat = output.reshape(-1, output.size(-1))
            target_flat = target.reshape(-1)
            loss = criterion(output_flat, target_flat)
            
            # Metrics
            predictions = output.argmax(dim=-1)
            token_acc = token_accuracy(predictions, target, tokenizer.pad_idx)
            seq_acc = sequence_accuracy(predictions, target, tokenizer.pad_idx)
            f1 = compute_f1_score(predictions, target, tokenizer.pad_idx)
            
            total_loss += loss.item()
            total_token_acc += token_acc
            total_seq_acc += seq_acc
            total_f1 += f1
            num_batches += 1
            
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'tok_acc': f'{token_acc:.4f}',
                'seq_acc': f'{seq_acc:.4f}'
            })
    
    return {
        'loss': total_loss / num_batches,
        'token_accuracy': total_token_acc / num_batches,
        'seq_accuracy': total_seq_acc / num_batches,
        'f1_score': total_f1 / num_batches
    }


# Training loop
print("Starting training...\n")
best_val_acc = 0.0  # Track best validation sequence accuracy
history = {
    'train_loss': [], 'train_token_acc': [], 'train_seq_acc': [], 'train_f1': [],
    'val_loss': [], 'val_token_acc': [], 'val_seq_acc': [], 'val_f1': [],
    'test_loss': [], 'test_token_acc': [], 'test_seq_acc': [], 'test_f1': []
}

for epoch in range(TRANSFORMER_CONFIG['num_epochs']):
    print(f"\nEpoch {epoch + 1}/{TRANSFORMER_CONFIG['num_epochs']}")
    print("-" * 70)
    
    start_time = time.time()
    
    # Train (with teacher forcing)
    train_metrics = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Validate (using greedy generation)
    val_metrics = evaluate(model, val_loader, criterion, device, tokenizer)
    
    # Test evaluation (using greedy generation)
    test_metrics = evaluate(model, test_loader, criterion, device, tokenizer)
    
    epoch_time = time.time() - start_time
    
    # Update history
    history['train_loss'].append(train_metrics['loss'])
    history['train_token_acc'].append(train_metrics['token_accuracy'])
    history['train_seq_acc'].append(train_metrics['seq_accuracy'])
    history['train_f1'].append(train_metrics['f1_score'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_token_acc'].append(val_metrics['token_accuracy'])
    history['val_seq_acc'].append(val_metrics['seq_accuracy'])
    history['val_f1'].append(val_metrics['f1_score'])
    history['test_loss'].append(test_metrics['loss'])
    history['test_token_acc'].append(test_metrics['token_accuracy'])
    history['test_seq_acc'].append(test_metrics['seq_accuracy'])
    history['test_f1'].append(test_metrics['f1_score'])
    
    # Print metrics
    print(f"Train - Loss: {train_metrics['loss']:.4f} | Token Acc: {train_metrics['token_accuracy']:.4f} | Seq Acc: {train_metrics['seq_accuracy']:.4f} | F1: {train_metrics['f1_score']:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f} | Token Acc: {val_metrics['token_accuracy']:.4f} | Seq Acc: {val_metrics['seq_accuracy']:.4f} | F1: {val_metrics['f1_score']:.4f}")
    print(f"Test  - Loss: {test_metrics['loss']:.4f} | Token Acc: {test_metrics['token_accuracy']:.4f} | Seq Acc: {test_metrics['seq_accuracy']:.4f} | F1: {test_metrics['f1_score']:.4f}")
    print(f"Time: {epoch_time:.2f}s")
    
    # Save best model (based on validation sequence accuracy)
    if val_metrics['seq_accuracy'] > best_val_acc:
        best_val_acc = val_metrics['seq_accuracy']
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_seq_acc': val_metrics['seq_accuracy'],
            'val_token_acc': val_metrics['token_accuracy'],
            'tokenizer': tokenizer,
            'config': TRANSFORMER_CONFIG
        }
        torch.save(checkpoint, f"{TRAIN_CONFIG['checkpoint_dir']}/best_model.pt")
        print(f"✓ Saved best model (Val Seq Acc: {val_metrics['seq_accuracy']:.4f})")
    
    # Save checkpoint periodically
    if (epoch + 1) % TRAIN_CONFIG['save_every'] == 0:
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'tokenizer': tokenizer,
            'config': TRANSFORMER_CONFIG
        }
        torch.save(checkpoint, f"{TRAIN_CONFIG['checkpoint_dir']}/checkpoint_epoch_{epoch+1}.pt")

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Best validation sequence accuracy: {best_val_acc:.4f}")

## 11. Training/Validation/Test Plots

Plot loss and accuracy curves for train/val/test sets across all epochs.

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss plot
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2, marker='o')
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2, marker='s')
axes[0, 0].plot(epochs_range, history['test_loss'], 'g-', label='Test Loss', linewidth=2, marker='^')
axes[0, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Training/Validation/Test Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Token accuracy
axes[0, 1].plot(epochs_range, history['train_token_acc'], 'b-', label='Train Token Acc', linewidth=2, marker='o')
axes[0, 1].plot(epochs_range, history['val_token_acc'], 'r-', label='Val Token Acc', linewidth=2, marker='s')
axes[0, 1].plot(epochs_range, history['test_token_acc'], 'g-', label='Test Token Acc', linewidth=2, marker='^')
axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Token Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

# Sequence accuracy
axes[1, 0].plot(epochs_range, history['train_seq_acc'], 'b-', label='Train Seq Acc', linewidth=2, marker='o')
axes[1, 0].plot(epochs_range, history['val_seq_acc'], 'r-', label='Val Seq Acc', linewidth=2, marker='s')
axes[1, 0].plot(epochs_range, history['test_seq_acc'], 'g-', label='Test Seq Acc', linewidth=2, marker='^')
axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Sequence Accuracy (Exact Match)', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# F1 score
axes[1, 1].plot(epochs_range, history['train_f1'], 'b-', label='Train F1', linewidth=2, marker='o')
axes[1, 1].plot(epochs_range, history['val_f1'], 'r-', label='Val F1', linewidth=2, marker='s')
axes[1, 1].plot(epochs_range, history['test_f1'], 'g-', label='Test F1', linewidth=2, marker='^')
axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('F1 Score', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Token-Level F1 Score', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('training_plots_transformer.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ All training curves saved!")
print("  - training_plots_transformer.png")
print("\nNote: All plots now show Train/Val/Test metrics across epochs")

## 12. Visualize Predictions on Random Validation Mazes

Visualize the model's predictions on 5 randomly selected mazes from the validation set.

In [ ]:
import random
import re
import numpy as np

# Define maze plotting functions
def parse_coords(s):
    nums = re.findall(r"-?\d+", s)
    return tuple(map(int, nums)) if len(nums) == 2 else None

def extract_between(tag, text):
    """Extract content between start and end tags"""
    patterns = [
        rf"<\s*{tag}\s*[_\-\s]?\s*START\s*>(.*?)<\s*{tag}\s*[_\-\s]?\s*END\s*>",
        rf"<\s*{tag}START\s*>(.*?)<\s*{tag}END\s*>",
        rf"<\s*{tag}\s*START\s*>(.*?)<\s*{tag}\s*END\s*>",
        rf"<\s*{tag.replace(' ', '_')}\s*START\s*>(.*?)<\s*{tag.replace(' ', '_')}\s*END\s*>",
    ]
    for p in patterns:
        m = re.search(p, text, re.S | re.I)
        if m:
            return m.group(1).strip()
    raise ValueError(f"Could not find section for tag '{tag}'.")

def plot_maze(tokens, title="Maze", predicted_path=None):
    """Plot maze with optional predicted path overlay"""
    text = " ".join(tokens)
    adj_section = extract_between("ADJLIST", text)
    origin_section = extract_between("ORIGIN", text)
    target_section = extract_between("TARGET", text)
    path_section = extract_between("PATH", text)

    origin = parse_coords(origin_section)
    target = parse_coords(target_section)

    # Parse edges
    edge_matches = re.findall(r"\(\s*-?\d+\s*,\s*-?\d+\s*\)\s*<-->\s*\(\s*-?\d+\s*,\s*-?\d+\s*\)", adj_section)
    edges = []
    for em in edge_matches:
        coords = re.findall(r"\(\s*-?\d+\s*,\s*-?\d+\s*\)", em)
        a = parse_coords(coords[0])
        b = parse_coords(coords[1])
        edges.append((a, b))

    # Parse ground truth path
    path = [parse_coords(p) for p in re.findall(r"\(\s*-?\d+\s*,\s*-?\d+\s*\)", path_section)]
    if not path:
        nums = re.findall(r"-?\d+\s*,\s*-?\d+", path_section)
        path = [tuple(map(int, re.findall(r"-?\d+", s))) for s in nums]

    if not edges:
        raise ValueError("No edges found in adjacency list.")

    rows, cols = 6, 6
    vertical_walls = np.ones((rows, cols + 1), dtype=bool)
    horizontal_walls = np.ones((rows + 1, cols), dtype=bool)

    for (r1, c1), (r2, c2) in edges:
        if r1 == r2:
            c_between = min(c1, c2) + 1
            vertical_walls[r1, c_between] = False
        elif c1 == c2:
            r_between = min(r1, r2) + 1
            horizontal_walls[r_between, c1] = False

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')

    # Draw grid
    for r in range(rows):
        for c in range(cols):
            x0, x1 = c, c + 1
            y_top = rows - r
            y_bot = rows - r - 1
            ax.plot([x0, x1], [y_top, y_top], color='lightgray', lw=2)
            ax.plot([x0, x1], [y_bot, y_bot], color='lightgray', lw=2)
            ax.plot([x0, x0], [y_bot, y_top], color='lightgray', lw=2)
            ax.plot([x1, x1], [y_bot, y_top], color='lightgray', lw=2)

    # Draw walls
    for r in range(rows):
        for c in range(cols + 1):
            if vertical_walls[r, c]:
                x = c
                y_top = rows - r
                y_bot = rows - r - 1
                ax.plot([x, x], [y_bot, y_top], color='black', lw=5, solid_capstyle='butt')

    for r in range(rows + 1):
        for c in range(cols):
            if horizontal_walls[r, c]:
                y = rows - r
                ax.plot([c, c + 1], [y, y], color='black', lw=5, solid_capstyle='butt')

    # Shade ground truth path cells
    if path:
        for (r, c) in path:
            x0, x1 = c, c + 1
            y_top = rows - r
            y_bot = rows - r - 1
            rect = plt.Rectangle((x0, y_bot), 1, 1, facecolor=(0.9, 1, 0.9), edgecolor=None, zorder=0)
            ax.add_patch(rect)

    # Draw ground truth path (green)
    if path:
        path_x = [c + 0.5 for (r, c) in path]
        path_y = [rows - r - 0.5 for (r, c) in path]
        ax.plot(path_x, path_y, linestyle='-', linewidth=2, color='green', label='Ground Truth', zorder=3)
        ax.scatter(path_x[0], path_y[0], c='green', s=100, marker='o', zorder=5)
        ax.scatter(path_x[-1], path_y[-1], c='green', s=100, marker='x', zorder=5)

    # Draw predicted path (red) if provided
    if predicted_path:
        pred_x = [c + 0.5 for (r, c) in predicted_path]
        pred_y = [rows - r - 0.5 for (r, c) in predicted_path]
        ax.plot(pred_x, pred_y, linestyle='--', linewidth=2, color='red', label='Predicted', zorder=4, alpha=0.7)
        ax.scatter(pred_x[0], pred_y[0], c='red', s=80, marker='o', zorder=5, alpha=0.7)
        ax.scatter(pred_x[-1], pred_y[-1], c='red', s=80, marker='x', zorder=5, alpha=0.7)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.set_title(title, fontsize=12, fontweight='bold')
    if predicted_path:
        ax.legend()
    plt.tight_layout()
    plt.show()


# Load best model
checkpoint = torch.load(f"{TRAIN_CONFIG['checkpoint_dir']}/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Visualizing Predictions on 5 Random Validation Mazes")
print("=" * 80)

# Sample 5 random indices
val_indices = random.sample(range(len(val_subset)), 5)

for i, idx in enumerate(val_indices, 1):
    # Get the actual sample from the original dataset
    actual_idx = val_subset.indices[idx]
    sample = train_dataset.df.iloc[actual_idx]
    
    input_sequence = sample['input_sequence']
    target_path = sample['output_path']
    maze_type = sample['maze_type']
    
    # Encode input
    input_indices = tokenizer.encode(input_sequence)
    input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)
    
    # Generate prediction
    with torch.no_grad():
        predicted_seq = model.generate(
            input_tensor,
            max_len=200,
            sos_idx=tokenizer.sos_idx,
            eos_idx=tokenizer.eos_idx
        )
    
    # Decode prediction
    predicted_tokens_raw = tokenizer.decode(predicted_seq[0].cpu().tolist())
    predicted_tokens_raw = [t for t in predicted_tokens_raw if t not in ['<PAD>', '<SOS>', '<EOS>', '<UNK>']]
    
    # Truncate at <EOS> if present (though already filtered)
    predicted_tokens = predicted_tokens_raw
    
    # Extract predicted path coordinates
    predicted_path_coords = []
    for token in predicted_tokens:
        coord = parse_coords(token)
        if coord is not None:
            predicted_path_coords.append(coord)
    
    # Check if prediction matches target
    is_correct = (predicted_tokens == target_path)
    
    # Calculate token overlap
    target_set = set(target_path)
    pred_set = set(predicted_tokens)
    overlap = len(target_set & pred_set)
    token_acc = overlap / max(len(target_set), 1)
    
    # Create title with metrics
    status = "✓ CORRECT" if is_correct else "✗ INCORRECT"
    title = f"Maze {i} ({maze_type}) - {status}\nToken Overlap: {overlap}/{len(target_set)} ({token_acc:.1%})"
    
    print(f"\nMaze {i}:")
    print(f"  Type: {maze_type}")
    print(f"  Status: {status}")
    print(f"  Target Length: {len(target_path)}, Predicted Length: {len(predicted_tokens)}")
    print(f"  Token Overlap: {overlap}/{len(target_set)} ({token_acc:.1%})")
    
    # Plot the maze with both paths
    plot_maze(input_sequence + target_path, title=title, predicted_path=predicted_path_coords if predicted_path_coords else None)

print("\n" + "=" * 80)
print("Visualization complete!")

## 13. Test Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(f"{TRAIN_CONFIG['checkpoint_dir']}/best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print("Evaluating on test set...\n")

# Evaluate with greedy generation
test_metrics = evaluate(model, test_loader, criterion, device, tokenizer)

print("="*70)
print("TEST SET RESULTS (Greedy Generation)")
print("="*70)
print(f"Test Token Accuracy: {test_metrics['token_accuracy']:.4f} ({test_metrics['token_accuracy']*100:.2f}%)")
print(f"Test Sequence Accuracy: {test_metrics['seq_accuracy']:.4f} ({test_metrics['seq_accuracy']*100:.2f}%)")
print(f"Test F1-Score: {test_metrics['f1_score']:.4f}")
print("="*70)

# Detailed evaluation
print("\nDetailed generation metrics...\n")
model.eval()
correct_sequences = 0
correct_tokens = 0
total_tokens = 0
total_sequences = 0
total_f1 = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating"):
        src = batch['src'].to(device)
        target = batch['target']
        
        predictions = model.generate(
            src,
            max_len=200,
            sos_idx=tokenizer.sos_idx,
            eos_idx=tokenizer.eos_idx
        )
        
        for pred, tgt in zip(predictions, target):
            pred_tokens = pred[1:].cpu().tolist()  # Remove SOS
            tgt_tokens = tgt.tolist()
            
            # Truncate at EOS/PAD
            def truncate(tokens):
                result = []
                for t in tokens:
                    if t == tokenizer.eos_idx or t == tokenizer.pad_idx:
                        break
                    result.append(t)
                return result
            
            pred_clean = truncate(pred_tokens)
            tgt_clean = truncate(tgt_tokens)
            
            # Sequence accuracy
            if pred_clean == tgt_clean:
                correct_sequences += 1
            
            # Token accuracy
            for p, t in zip(pred_clean, tgt_clean):
                if p == t:
                    correct_tokens += 1
                total_tokens += 1
            
            # Account for length mismatch
            total_tokens += abs(len(pred_clean) - len(tgt_clean))
            
            # F1 score
            if len(tgt_clean) > 0:
                pred_set = set(pred_clean)
                tgt_set = set(tgt_clean)
                tp = len(pred_set & tgt_set)
                fp = len(pred_set - tgt_set)
                fn = len(tgt_set - pred_set)
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                total_f1 += f1
            
            total_sequences += 1

generation_seq_accuracy = correct_sequences / total_sequences
generation_token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0
avg_f1 = total_f1 / total_sequences

print("\n" + "="*70)
print("TEST SET RESULTS (Greedy Generation)")
print("DETAILED TEST METRICS")
print(f"Token Accuracy: {generation_token_accuracy:.4f} ({generation_token_accuracy*100:.2f}%)")
print(f"Sequence Accuracy: {generation_seq_accuracy:.4f} ({generation_seq_accuracy*100:.2f}%)")
print(f"F1-Score: {avg_f1:.4f}")
print(f"Correct sequences: {correct_sequences}/{total_sequences}")
print(f"Correct tokens: {correct_tokens}/{total_tokens}")
print("="*70)

## 14. Download Trained Model

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # Download best model
    model_path = f"{TRAIN_CONFIG['checkpoint_dir']}/best_model.pt"
    print(f"Downloading: {model_path}")
    files.download(model_path)
    
    # Download plots
    if os.path.exists('training_plots_transformer.png'):
        files.download('training_plots_transformer.png')
    
    print("\n✓ Download complete!")
else:
    print("Not on Colab. Files saved locally.")
    print(f"Model: {TRAIN_CONFIG['checkpoint_dir']}/best_model.pt")